# When integers meet floats
**Python Foundation · Type promotion and division**

In Google Colab, choose **Runtime → Run all**. Connect to a Python runtime if prompted; Google sign-in may be needed. Then try the interactive calculator below. Every code cell is editable.

```python
10 + 10.1       # 20.1, a float
123 + 5 / 5     # 124.0, a float
```

For ordinary arithmetic with built-in `int` and `float` values, mixed operands use floating-point arithmetic. For this first example, think of `10 + 10.1` as `10.0 + 10.1`. The original integer variable does not change its type.

In [ ]:
a = 10
b = 10.1
result = a + b
for name, value in [("a", a), ("b", b), ("a + b", result)]:
    print(f"{name:7} = {value!r:6} | {type(value).__name__}")

## Division comes first

`/` has higher precedence than `+`. With these built-in real numbers, true division returns a `float`, even when the quotient is a whole number.

| Step | Expression | Value | Type |
|---|---|---|---|
| 1 | `5 / 5` | `1.0` | `float` |
| 2 | `123 + 1.0` | `124.0` | `float` |

Compare `/` with `//`: floor division of two integers returns an integer. With a float operand, floor division returns a float. Floor division rounds toward negative infinity, not toward zero.

In [ ]:
part = 5 / 5
result = 123 + part
print("5 / 5:", repr(part), type(part).__name__)
print("123 + 5 / 5:", repr(result), type(result).__name__)
for value in [5 // 5, 5.0 // 5, -5 // 2]:
    print(repr(value), type(value).__name__)

## Try the type pipeline

Enter numeric literals such as `10`, `10.0`, or `10.1`. A decimal point or exponent notation makes a float literal. Choose an operator and press **Calculate**. Presets demonstrate mixed addition, true division, and floor division.

The pipeline labels the original operands, the arithmetic type, and the actual Python result. Invalid input and division by zero produce an explanation rather than stopping the explorer.

In [ ]:
import ast
import html
import math
import operator
import ipywidgets as widgets
from IPython.display import display
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

OPERATIONS = {"+": operator.add, "-": operator.sub, "*": operator.mul,
              "/": operator.truediv, "//": operator.floordiv}

def number_literal(text):
    value = ast.literal_eval(text.strip())
    if type(value) not in (int, float):
        raise ValueError("Use an int or float literal, such as 10 or 10.1.")
    if isinstance(value, float) and not math.isfinite(value):
        raise ValueError("Use a finite number.")
    return value

def calculate(left, symbol, right):
    a, b = number_literal(left), number_literal(right)
    return a, b, OPERATIONS[symbol](a, b)

def render_pipeline(a, symbol, b, result):
    mixed = type(a) is not type(b)
    floating = isinstance(a, float) or isinstance(b, float)
    if floating:
        middle = f"Floating-point arithmetic: {float(a)!r} {symbol} {float(b)!r}"
        explanation = "The integer operand is promoted for this calculation." if mixed else "Both operands are already floats."
    elif symbol == "/":
        middle = "True division of two integers"
        explanation = "The / operator produces a float quotient for these real-number operands."
    else:
        middle = "Integer arithmetic"
        explanation = "Both operands are ints; this operation keeps an int result."
    def tile(label, value, kind):
        return f'<div style="padding:16px;border:1px solid #cadce5;border-radius:10px;flex:1;min-width:150px"><small>{label}</small><div style="font-size:23px;overflow-wrap:anywhere">{html.escape(repr(value))}</div><b style="color:#237a99">{kind}</b></div>'
    return f"""<div style="padding:20px;background:#fff;color:#172b3a;border:1px solid #cbd5e1;border-radius:14px;font:15px/1.7 system-ui">
    <div style="display:flex;gap:12px;flex-wrap:wrap">{tile('Left operand',a,type(a).__name__)}{tile('Right operand',b,type(b).__name__)}</div>
    <p style="border-left:3px solid #237a99;padding-left:13px">{html.escape(middle)}<br>{explanation}</p>
    {tile('Python result',result,type(result).__name__)}
    <p>The original operands remain {type(a).__name__} and {type(b).__name__}.</p></div>"""

left_input = widgets.Text(value="10", description="Left:")
right_input = widgets.Text(value="10.1", description="Right:")
op_input = widgets.Dropdown(options=list(OPERATIONS), value="+", description="Operator:")
run_button = widgets.Button(description="Calculate", button_style="info")
presets = widgets.Dropdown(options=[("Mixed addition", ("10", "+", "10.1")),
    ("Integer addition", ("10", "+", "10")),
    ("True division", ("5", "/", "5")),
    ("Integer floor division", ("5", "//", "5")),
    ("Float floor division", ("5.0", "//", "5"))], description="Preset:")
view = widgets.HTML()

def update(button=None):
    try:
        a, b, result = calculate(left_input.value, op_input.value, right_input.value)
        view.value = render_pipeline(a, op_input.value, b, result)
    except (ValueError, SyntaxError, TypeError, ZeroDivisionError, OverflowError) as error:
        view.value = '<p style="padding:16px;color:#a43b31">' + html.escape(type(error).__name__ + ": " + str(error)) + '</p>'

def load_preset(change):
    left_input.value, op_input.value, right_input.value = change["new"]
    update()

run_button.on_click(update)
presets.observe(load_preset, names="value")
update()
display(widgets.VBox([presets, left_input, op_input, right_input, run_button, view]))

## A short prediction challenge

Change the values below. Predict both the **value** and the **type**, then run the cell. Can you make the final result an `int` by replacing `/` with `//`?

In [ ]:
base = 123
numerator = 5
denominator = 5
quotient = numerator / denominator
answer = base + quotient
print(f"First: {numerator!r} / {denominator!r} = {quotient!r} ({type(quotient).__name__})")
print(f"Then: {base!r} + {quotient!r} = {answer!r} ({type(answer).__name__})")

## Precision: a small but useful distinction

`10` is exactly representable as `10.0`. That does **not** mean every integer or decimal fraction is exactly representable as a float. Most decimal fractions, including `10.1`, use a nearby binary approximation. Python's printed `20.1` is a short representation of the stored float.

Large integers can also lose precision when converted to float. This is separate from deciding the result's type.

In [ ]:
print("A familiar rounding example:", 0.1 + 0.2)
large = 2**53 + 1
print("Original integer:", large)
print("After float conversion:", float(large))
print("Exact round trip?", int(float(large)) == large)

## Remember

- For these built-in numeric types, mixed `int`/`float` arithmetic uses floats.
- `/` produces a float quotient here, even when both inputs are ints.
- `//` floors the quotient; its result type depends on the operand types.
- Promotion for an operation does not turn the original integer variable into a float.

Sources: [Python numeric types](https://docs.python.org/3/library/stdtypes.html#numeric-types-int-float-complex), [Division operators](https://docs.python.org/3/reference/expressions.html#binary-arithmetic-operations), [Floating-point arithmetic](https://docs.python.org/3/tutorial/floatingpoint.html).